# kgat on Colab — mine → SFT → eval → GRPO

Trains the budget-adaptive traversal controller (Qwen3-0.6B, QLoRA) on WebQSP.
Runtime: **T4 (free)** is enough for mining + SFT + eval and short GRPO runs;
use an A100/L4 for the full GRPO λ-sweep.

Each stage writes under `outputs/` — mount Drive (last cell) if you want
checkpoints to survive session resets.

In [ ]:
!nvidia-smi

In [ ]:
# --- get the code ---------------------------------------------------------
# Option A: clone your repo (set the URL). Option B: upload a zip of kgat/ and unzip.
REPO_URL = ""  # e.g. "https://github.com/<you>/kgat.git"

import os

if REPO_URL:
    !git clone {REPO_URL} kgat
elif not os.path.exists("kgat"):
    from google.colab import files

    print("Upload a zip of the kgat repo...")
    uploaded = files.upload()
    !unzip -q {list(uploaded)[0]} -d .
%cd kgat
# torch is preinstalled on Colab; [colab] adds transformers/peft/accelerate/bnb/datasets
!pip install -q -e ".[colab]"
!pytest -q  # foundation must be green before burning GPU time

In [ ]:
# --- 1. download WebQSP (preprocessed per-question subgraphs) ---------------
!bash scripts/download_data.sh webqsp
!ls -la data/webqsp/

In [ ]:
# --- 2. mine oracle trajectories (M3) — CPU-only, a few minutes -------------
!python -m kgat.train.mine_trajectories dataset=webqsp dataset.split=train

In [ ]:
# --- 3. SFT the 0.6B controller (M4) — QLoRA, fits a T4 ---------------------
# First run: add train.sft.max_examples=2000 for a ~20-min sanity pass before
# committing to the full set.
!python -m kgat.train.sft train=sft dataset=webqsp dataset.split=train model=qwen3-0.6b \
    train.sft.gradient_checkpointing=true

In [ ]:
# --- 4. evaluate: dummy baseline vs the SFT'd controller --------------------
# dataset.limit keeps eval snappy; drop it for full-split numbers.
!python -m kgat.eval.harness dataset=webqsp dataset.split=test dataset.limit=200 \
    controller=dummy synth=dummy run_label=webqsp_dummy paths.output_dir=outputs/colab
!python -m kgat.eval.harness dataset=webqsp dataset.split=test dataset.limit=200 \
    model=qwen3-0.6b controller=decoder synth=dummy \
    controller.adapter_path=outputs/adapters/qwen3-0.6b-sft \
    run_label=webqsp_sft paths.output_dir=outputs/colab

In [ ]:
# --- 5. GRPO (M5) — smoke first, then the real run --------------------------
# Smoke (~minutes): a few updates to confirm rewards/gradients flow on GPU.
!python -m kgat.train.grpo dataset=webqsp dataset.split=train model=qwen3-0.6b \
    train.grpo.max_questions=32 train.grpo.group_size=4 train.grpo.lam=0.1

# Real run + frontier sweep (hours; A100 recommended) — uncomment:
# !python -m kgat.train.grpo -m dataset=webqsp dataset.split=train model=qwen3-0.6b \
#     train.grpo.lam=0.0,0.05,0.1,0.2,0.4

In [ ]:
# --- 6. eval each λ-adapter, then build the cost/quality frontier -----------
import glob, subprocess

for adapter in sorted(glob.glob("outputs/adapters/qwen3-0.6b-grpo-lam*")):
    lam = adapter.rsplit("lam", 1)[-1]
    subprocess.run(
        [
            "python",
            "-m",
            "kgat.eval.harness",
            "dataset=webqsp",
            "dataset.split=test",
            "dataset.limit=200",
            "model=qwen3-0.6b",
            "controller=decoder",
            "synth=dummy",
            f"controller.adapter_path={adapter}",
            f"run_label=webqsp_grpo_lam{lam}",
            "paths.output_dir=outputs/colab",
        ],
        check=True,
    )

!python -m kgat.eval.frontier --runs-dir outputs/colab --out-dir outputs/colab
from IPython.display import Image

Image("outputs/colab/frontier.png")

In [ ]:
# --- optional: persist adapters + results to Drive ---------------------------
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p /content/drive/MyDrive/kgat && cp -r outputs /content/drive/MyDrive/kgat/